In [ ]:
import numpy as np
import pandas as pd

from uncertaintyx.fit.randomsampling import Bootstrap
from uncertaintyx.fit.regression import HomoscedasticRegression
from uncertaintyx.interface.core import M
from uncertaintyx.plot.plots import MatrixPlot
from uncertaintyx.plot.plots import RegressionPlot

In [ ]:
class M2(M):
    """
    Empirical model function to fit data in Lee et al. (2010, Figure 2,
    https://www.ioccg.org/groups/Software_OCA/QAA_v5.pdf).
    """

    def eval(self, par: np.ndarray, x: np.ndarray) -> np.ndarray:
        a, b = par
        term = np.exp(-x)
        return a * (1.0 - b * term)

    def jac_p(self, par: np.ndarray, x: np.ndarray) -> np.ndarray:
        a, b = par
        term = np.exp(-x)
        return np.stack([1.0 - b * term, -a * term], axis=-1)

    def jac_x(self, par: np.ndarray, x: np.ndarray) -> np.ndarray:
        a, b = par
        term = np.exp(-x)
        return a * b * term

    def estimate(
        self, x: np.ndarray | None = None, y: np.ndarray | None = None
    ) -> np.ndarray:
        return np.array([2.0, 1.2])

    @property
    def f():
        return self.eval

In [ ]:
class M3(M):
    """
    Empirical model function to fit data in Lee et al. (2010, Figure 3,
    https://www.ioccg.org/groups/Software_OCA/QAA_v5.pdf).
    """

    def eval(self, par, x: np.ndarray) -> np.ndarray:
        a, b, c = par
        term = 1.0 / (c + x)
        return a + b * term

    def jac_p(self, par: np.ndarray, x: np.ndarray) -> np.ndarray:
        _, b, c = par
        term = 1.0 / (c + x)
        return np.stack([np.ones_like(term), term, -b * term * term], axis=-1)

    def jac_x(self, par: np.ndarray, x: np.ndarray) -> np.ndarray:
        _, b, c = par
        term = 1.0 / (c + x)
        return -b * term * term

    def estimate(
        self, x: np.ndarray | None = None, y: np.ndarray | None = None
    ) -> np.ndarray:
        return np.array([0.015, 0.002, 0.6])

    @property
    def f():
        return self.eval

# Figure 2

In [ ]:
df = pd.read_csv("../test/resources/fig2.csv", sep=";")
x = df["X"].values
y = df["Y"].values

In [ ]:
result = Bootstrap(HomoscedasticRegression()).fit(M2(), x, y)

In [ ]:
print("fit parameters = ", result.popt)
print("fit parameters uncertainty = ", result.punc)
print("fit parameters covariance matrix = ", result.pcov)
print("fit residual variance = ", result.rvar)

In [ ]:
RegressionPlot(result).plot(
    x,
    y,
    title="Replica of Lee et al. (2010, Figure 2)",
    xlabel=r"$r(443~\mathrm{nm})$ / $r(555~\mathrm{nm})$",
    ylabel=r"$\eta$",
    xrange=(0.5, 4.5),
    yrange=(-0.5, 2.5),
)

In [ ]:
MatrixPlot().plot(
    result.ycov_p(np.linspace(0.5, 4.5, 1000)),
    xlabel=r"$r(443~\mathrm{nm})$ / $r(555~\mathrm{nm})$",
    ylabel=r"$r(443~\mathrm{nm})$ / $r(555~\mathrm{nm})$",
    xrange=(0.5, 4.5),
    yrange=(0.5, 4.5),
    cmap="cividis",
    cbar_label=r"variance-covariance $U_p(\eta)$",
    cbar_max=0.024,
    cbar_min=-0.024,
)

# Figure 3

In [ ]:
df = pd.read_csv("../test/resources/fig3.csv", sep=";")
x = df["X"].values
y = df["Y"].values

In [ ]:
result = Bootstrap(HomoscedasticRegression()).fit(M3(), x, y)

In [ ]:
print("fit parameters = ", result.popt)
print("fit parameters uncertainty = ", result.punc)
print("fit parameters covariance matrix = ", result.pcov)
print("fit residual variance = ", result.rvar)

In [ ]:
RegressionPlot(result).plot(
    x,
    y,
    title="Replica of Lee et al. (2010, Figure 3)",
    xlabel=r"$r(443~\mathrm{nm})$ / $r(555~\mathrm{nm})$",
    ylabel=r"S ($\mathrm{nm}^{-1}$)",
    xrange=(0.050, 9.950),
    yrange=(0.005, 0.035),
)

In [ ]:
MatrixPlot().plot(
    result.ycov_p(np.linspace(0.05, 9.95, 1000)),
    xlabel=r"$r(443~\mathrm{nm})$ / $r(555~\mathrm{nm})$",
    ylabel=r"$r(443~\mathrm{nm})$ / $r(555~\mathrm{nm})$",
    xrange=(0.05, 9.95),
    yrange=(0.05, 9.95),
    cmap="viridis",
    cbar_label=r"variance-covariance $U_p(S)$ ($\mathrm{nm}^{-2}$)",
    cbar_max=3.0e-05,
    cbar_min=0.0e-05,
)